# Propagation

> Where a transmission's channel coefficient h comes from: fading models and ray-traced tables.

In [ ]:
#| default_exp propagation

In [ ]:
#| hide
from nbdev.showdoc import *

A `Fading` gives the channel coefficient `h` of a transmission. `h` includes the path loss (`|h|²` is the power
gain) and the phase. Channels at both levels use it: `SystemLevelChannel` (see `system`) and `LinkLevelChannel`
(see `link`).

* `NoFading`: a constant `h`.
* `RayleighFading`: `h ~ CN(0, gain)`, drawn anew for every transmission.
* `RayTracedFading`: `h` looked up in a table ray-traced beforehand (e.g. with Sionna RT), at the nodes' positions
  (discrete environments: the nodes are on the table's points).
* `RayTracedPathsFading`: paths ray-traced at anchor points (e.g. every 5 cm), moved to the nodes' exact
  positions (continuous environments).

In [ ]:
#| export
from __future__ import annotations

import bisect
import math
import warnings
from abc import ABC, abstractmethod
from typing import Mapping, Sequence

import numpy as np

from comm_core.core import NodeId, Transmission
from comm_core.topology import Link

In [ ]:
#| export
class Fading(ABC):
    "The channel coefficient h of a transmission."

    @abstractmethod
    def coefficient(
        self,
        transmission: Transmission, # The transmission (its sender and receiver)
        link: Link, # Its link
    ) -> complex:
        "h of this transmission (math.inf: the nodes are at the same place, which means a perfect link)."

    def response(
        self,
        transmission: Transmission, # The transmission (its sender and receiver)
        link: Link, # Its link
    ) -> np.ndarray:
        "h at each subcarrier (a frequency-selective fading); by default flat: one value, `coefficient`."
        return np.array([complex(self.coefficient(transmission, link))])

    def reset(self, seed: int | None = None) -> None:
        "Start over; `seed` reseeds the fading's random generator, if it has one."
        pass


class NoFading(Fading):
    "The same coefficient for every transmission."

    def __init__(
        self,
        h: complex = 1.0, # The coefficient
    ):
        self.h = h

    def coefficient(self, transmission, link):
        return self.h


class RayleighFading(Fading):
    "h ~ CN(0, gain), drawn anew for every transmission (constant during a packet)."

    def __init__(
        self,
        gain: float = 1.0, # Average power gain E|h|^2 (the path loss)
        seed: int | None = None, # Seed of the draws
    ):
        self.gain = gain
        self.rng = np.random.default_rng(seed)

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)

    def coefficient(self, transmission, link):
        return complex(self.rng.standard_normal(), self.rng.standard_normal()) * math.sqrt(self.gain / 2)

In [ ]:
show_doc(Fading.coefficient)

In [ ]:
t = Transmission(message=None, sender='a', receiver='b', start_time=0.0)
f = RayleighFading(gain=2.0, seed=0)
hs = np.array([f.coefficient(t, None) for _ in range(20_000)])
assert abs((np.abs(hs) ** 2).mean() - 2.0) < 0.05
f.reset(seed=0); assert f.coefficient(t, None) == hs[0]
assert NoFading(0.5j).coefficient(t, None) == 0.5j

### Time-correlated impairments

A fading can depend on time: a transmission carries its time (`transmission.start_time`, the network's time, or
the time of each attempt in a `FragmentingChannel`).

`GilbertElliottFading` uses it to weaken another fading by `penalty_db` during bad episodes. These are the episodes
of a two-state Markov process (good ↔ bad, with exponential durations), e.g. bursts of interference or a body
blocking the line of sight. The episodes are fixed by the seed, so two methods compared with the same seed see the
same bursts (common random numbers).

In [ ]:
#| export
class GilbertElliottFading(Fading):
    "Another fading, weakened by `penalty_db` during the bad episodes of a two-state Markov process in time."

    def __init__(
        self,
        fading: Fading, # The fading in good times
        mean_good_s: float = 3.0, # Mean time between bad episodes
        mean_bad_s: float = 0.6, # Mean length of a bad episode
        penalty_db: float = 30.0, # How much weaker the channel is during one (SINR drop)
        seed: int | None = None, # Seed of the episodes
    ):
        self.fading, self.mean_good_s, self.mean_bad_s, self.penalty_db = fading, mean_good_s, mean_bad_s, penalty_db
        self._start(seed)

    def _start(self, seed):
        self.rng, self.edges, self._end, self._bad = np.random.default_rng(seed), [], 0.0, False

    def _extend(self, until: float) -> None:
        while self._end <= until:                           # draw the episodes as far as needed (good at time 0)
            self._end += self.rng.exponential(self.mean_bad_s if self._bad else self.mean_good_s)
            self.edges.append(self._end)
            self._bad = not self._bad

    def bad(
        self,
        time: float, # A time
    ) -> bool:
        "Whether `time` is in a bad episode."
        self._extend(time)
        return bisect.bisect_right(self.edges, time) % 2 == 1

    def episodes(
        self,
        until: float, # Up to this time
    ) -> list:
        "The bad episodes up to `until`: [(start, end), ...]."
        self._extend(until)
        e = self.edges
        return [(e[i], min(e[i + 1], until)) for i in range(0, len(e) - 1, 2) if e[i] < until]

    def reset(self, seed=None):
        "`seed` draws new episodes (and reseeds the inner fading)."
        self.fading.reset(seed)
        if seed is not None:
            self._start(seed)

    def set_positions(self, positions) -> None:
        if hasattr(self.fading, 'set_positions'):
            self.fading.set_positions(positions)

    def _scale(self, transmission) -> float:
        return 10 ** (-self.penalty_db / 20) if self.bad(transmission.start_time) else 1.0

    def coefficient(self, transmission, link):
        h = self.fading.coefficient(transmission, link)
        return h if math.isinf(abs(h)) else h * self._scale(transmission)

    def response(self, transmission, link):
        h = self.fading.response(transmission, link)
        return h if np.isinf(h).any() else h * self._scale(transmission)

In [ ]:
show_doc(GilbertElliottFading.bad)

In [ ]:
show_doc(GilbertElliottFading.episodes)

In [ ]:
ge = GilbertElliottFading(NoFading(1.0), mean_good_s=3.0, mean_bad_s=0.6, penalty_db=30, seed=0)
times = np.arange(0, 2000, 0.05)
bad = np.array([ge.bad(x) for x in times])
assert not ge.bad(0.0) and abs(bad.mean() - 0.6 / 3.6) < 0.03             # bad 1/6 of the time
at = lambda x: Transmission(message=None, sender='a', receiver='b', start_time=x)
x_bad, x_good = times[bad][0], times[~bad][0]
assert abs(ge.coefficient(at(x_bad), None) - 10 ** -1.5) < 1e-12 and ge.coefficient(at(x_good), None) == 1.0
assert all(ge.bad((s + e) / 2) for s, e in ge.episodes(100))
same = GilbertElliottFading(NoFading(), seed=0)
assert same.episodes(500) == ge.episodes(500)                               # the same seed: the same bursts
ge.reset(seed=1)
assert ge.episodes(500) != same.episodes(500)

### Ray-traced tables

The coefficients between every pair of a set of positions, e.g. the CFR from Sionna RT's `paths.cfr(...)`, saved
as numpy arrays:

* `H[rx, tx]`: complex, `(N, N)` or `(N, N, num_subcarriers)`.
* `positions[i]`: the position of index `i`, e.g. a grid cell `(x, y)`.

Nodes move, so the table is told where they are (`set_positions`, e.g. at every step of an environment). A
transmission gets the `H[rx, tx]` of its nodes' positions. Two nodes at the same position have a perfect link.
With subcarriers, a transmission gets one coefficient, `√(mean |H|²)`: the bits are not spread over the subcarriers.

The coefficients must keep the path loss, so compute them with `paths.cfr(..., normalize=False)`. With
`normalize=True` every link has `|H|² = 1` (only the phases differ) and every link would be equally good. The table
warns when it is given such an array.

In [ ]:
#| export
def _key(position) -> tuple:
    return tuple(np.asarray(position).reshape(-1).tolist())


class RayTracedFading(Fading):
    "Coefficients ray-traced beforehand (e.g. Sionna RT) between a set of positions, looked up where the nodes are."

    def __init__(
        self,
        H, # Channel coefficients `H[rx, tx]` between the positions: (N, N) or (N, N, num_subcarriers), complex
        positions: Sequence, # The N positions (e.g. grid cells (x, y)), in the order of H's indices
        node_positions: Mapping | None = None, # Where the nodes are to begin with (see `set_positions`)
    ):
        H = np.asarray(H)
        if H.ndim == 3:
            gain = (np.abs(H) ** 2).mean(axis=-1)
        elif H.ndim == 2:
            gain = np.abs(H) ** 2
        else:
            raise ValueError(f"H: (N, N) or (N, N, num_subcarriers), not {H.shape}")
        if gain.shape[0] != gain.shape[1] or gain.shape[0] != len(positions):
            raise ValueError(f"H is {H.shape[:2]} for {len(positions)} positions")
        self.gain = gain.astype(np.float64)
        self.H = (H if H.ndim == 2 else np.sqrt(self.gain)).astype(np.complex128)   # one coefficient per link
        self.positions = [_key(p) for p in positions]
        self.index = {p: i for i, p in enumerate(self.positions)}
        off = ~np.eye(len(self.positions), dtype=bool)
        if off.any() and np.allclose(self.gain[off], 1.0):
            warnings.warn("every link has |H|^2 = 1: the coefficients were normalized (paths.cfr(normalize=True)), "
                          "so the path loss is lost and every link is equally good; use normalize=False")
        self.node_positions: dict = {}
        self.set_positions(node_positions or {})

    @classmethod
    def from_files(
        cls,
        channel_path: str, # .npy of H[rx, tx]
        positions_path: str, # .npy of the positions
        **kwargs, # The other arguments of `RayTracedFading`
    ) -> RayTracedFading:
        "Load the arrays saved after ray tracing."
        return cls(np.load(channel_path), np.load(positions_path), **kwargs)

    def set_positions(
        self,
        positions: Mapping[NodeId, Sequence], # Position of (some of) the nodes, e.g. {'agent_0': (3, 5)}
    ) -> None:
        "Where the nodes are now."
        for node, p in positions.items():
            self.node_positions[node] = _key(p)

    def _index(self, node: NodeId) -> int:
        if node not in self.node_positions:
            raise KeyError(f"no position for node {node!r}: call set_positions first")
        p = self.node_positions[node]
        if p not in self.index:
            raise KeyError(f"node {node!r} is at {p}, where no channel was ray-traced ({len(self.index)} positions)")
        return self.index[p]

    def path_gain(self, sender: NodeId, receiver: NodeId) -> float:
        "Linear path gain |H|^2 from `sender` to `receiver` at their current positions (inf if co-located)."
        tx, rx = self._index(sender), self._index(receiver)
        return math.inf if tx == rx else float(self.gain[rx, tx])

    def coefficient(self, transmission, link):
        "H[rx, tx] at the nodes' current positions (math.inf if co-located)."
        tx, rx = self._index(transmission.sender), self._index(transmission.receiver)
        return math.inf if tx == rx else complex(self.H[rx, tx])

In [ ]:
show_doc(RayTracedFading.from_files)

In [ ]:
show_doc(RayTracedFading.set_positions)

In [ ]:
show_doc(RayTracedFading.path_gain)

In [ ]:
import tempfile, os

In [ ]:
positions = [(0, 0), (1, 0), (5, 0)]
H = np.array([[0, 1e-3, 2e-5], [1e-3j, 0, 3e-5], [2e-5, 3e-5, 0]])
table = RayTracedFading(H, positions, node_positions={'a': (0, 0), 'b': (1, 0)})
assert table.coefficient(t, None) == 1e-3j and abs(table.path_gain('a', 'b') - 1e-6) < 1e-18
table.set_positions({'b': (0, 0)})
assert table.coefficient(t, None) == math.inf and table.path_gain('a', 'b') == math.inf     # same cell
table.set_positions({'b': (9, 9)})
try:
    table.coefficient(t, None); raise AssertionError('no error')
except KeyError as e:
    assert 'no channel was ray-traced' in str(e)
H3 = np.repeat(H[..., None], 4, axis=-1)
assert np.allclose(RayTracedFading(H3, positions).gain, table.gain)
with tempfile.TemporaryDirectory() as tmp:
    np.save(os.path.join(tmp, 'H.npy'), H); np.save(os.path.join(tmp, 'pos.npy'), np.array(positions))
    assert RayTracedFading.from_files(os.path.join(tmp, 'H.npy'), os.path.join(tmp, 'pos.npy')).positions == positions
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    RayTracedFading(np.exp(1j * np.ones((3, 3))), positions)
    assert any('normalize' in str(x.message) for x in w)

### Ray-traced paths: continuous positions

A table of channels at fixed points (`RayTracedFading`) suits discrete environments, where the nodes are always
on one of them. In continuous environments (robots, vehicles), a node is almost never exactly at a ray-traced point.
The channel cannot simply be taken from the nearest point either: its phase turns a full circle every wavelength
(5.8 cm at 5.18 GHz), so the nearest point's channel would be wrong for anything coherent.

`RayTracedPathsFading` keeps the *paths* ray-traced at anchor points, e.g. every 5 cm (see `grid_points`), instead
of their sum. Each path has a gain `a`, a delay `τ`, and the directions in which it leaves the transmitter
(`u_tx`) and reaches the receiver (`u_rx`). A node a little away from its anchor sees the same paths, moved:

* **The path as a line.** Unfolded at its reflections, a path is a straight line of length `L = c τ` that reaches
  the receiver from an image of the transmitter, in the direction `u_rx`.
* **Moving the ends.** Moving the receiver by `Δ_rx` and the transmitter by `Δ_tx` moves the ends of that line.
  The transmitter's image moves by `Δ_tx` mirrored, so that `u_tx` turns into the arrival direction. This gives
  the new length `L'`.
* **The moved path.** Its delay is `τ' = L' / c`, and its gain is `a · L / L' · exp(−j 2π f_c (τ' − τ))` (spreading
  over the longer or shorter path, and its phase).

The channel response at frequency `f` from the carrier is then `H(f) = Σ a e^{−j 2π f τ'}`.

* **Accuracy:** this is exact for line-of-sight paths and single reflections off flat surfaces, wherever the
  ends move. Every other path (several reflections, diffraction) is right to first order: the plane-wave
  approximation, `Δτ = −(u_tx·Δ_tx + u_rx·Δ_rx) / c`. It stays accurate while the paths themselves (which
  walls they hit) don't change between anchors, so the anchors can be much further apart than a wavelength. The
  tests below and the validation against Sionna RT in `sionna_rt` measure it.
* **Moving ends:** both can move, e.g. agent-to-agent links, with the transmitter and receiver anchors on the same
  set of points. A fixed access point is a single transmitter anchor.
* **Close nodes:** nodes on the same anchor (nearer each other than the anchors) have a perfect link, like nodes
  at the same place.
* **Direction:** links go both ways (`reciprocal`): a robot → access point link uses the access point → robot
  paths in reverse.
* **Data:** `trace_paths` (see `sionna_rt`) traces the anchors in a Sionna RT scene, and returns the arguments of
  `RayTracedPathsFading`.

In [ ]:
#| export
def grid_points(
    lo: Sequence[float], # Lower corner: (x, y) or (x, y, z)
    hi: Sequence[float], # Upper corner (included when the spacing divides the size)
    spacing: float, # Distance between neighbouring points, e.g. 0.05 (m)
    height: float | None = None, # z of the points of a 2-D grid (the antennas' height)
) -> np.ndarray:
    "Anchor points on a regular grid: (N, 3)."
    lo, hi = np.asarray(lo, float), np.asarray(hi, float)
    if lo.shape != hi.shape or lo.shape[0] not in (2, 3):
        raise ValueError("lo and hi: both (x, y) or both (x, y, z)")
    axes = [np.arange(l, h + spacing / 2, spacing) for l, h in zip(lo, hi)]
    points = np.stack(np.meshgrid(*axes, indexing='ij'), -1).reshape(-1, len(lo))
    if len(lo) == 2:
        if height is None:
            raise ValueError("a 2-D grid needs the antennas' height")
        points = np.concatenate([points, np.full((len(points), 1), float(height))], axis=1)
    return points

In [ ]:
g = grid_points((0, 0), (1, 0.5), 0.25, height=0.4)
assert g.shape == (5 * 3, 3) and (g[:, 2] == 0.4).all() and g[-1].tolist() == [1.0, 0.5, 0.4]
assert grid_points((0, 0, 0), (0.1, 0.1, 0.1), 0.05).shape == (27, 3)

In [ ]:
#| export
_C = 299_792_458.0                                     # speed of light (m/s)


class RayTracedPathsFading(Fading):
    "Paths ray-traced between anchor points, moved to the nodes' exact positions: continuous positions."

    def __init__(
        self,
        tx_points, # Transmitter anchors, (N_tx, 3)
        rx_points, # Receiver anchors, (N_rx, 3)
        a, # Complex gain of each path, (N_tx, N_rx, P); 0: no path
        tau, # Delay of each path (s), (N_tx, N_rx, P)
        u_tx, # Unit vector in which each path leaves the transmitter, (N_tx, N_rx, P, 3)
        u_rx, # Unit vector from the receiver towards where each path arrives from, (N_tx, N_rx, P, 3)
        frequency: float, # Carrier frequency (Hz)
        subcarriers=None, # Frequencies of the subcarriers relative to the carrier (Hz); None: the carrier alone
        reciprocal: bool = True, # Links go both ways: a path is also used from its receiver to its transmitter
        max_distance: float | None = None, # Farthest a node may be from its anchor; beyond it, an error (None: no limit)
        antenna_height: float | None = None, # z of the nodes given 2-D positions
        node_positions: Mapping | None = None, # Where the nodes are to begin with (see `set_positions`)
    ):
        self.tx_points = np.asarray(tx_points, np.float64).reshape(-1, 3)
        self.rx_points = np.asarray(rx_points, np.float64).reshape(-1, 3)
        self.a = np.asarray(a, np.complex128)
        self.tau = np.asarray(tau, np.float64)
        self.u_tx = np.asarray(u_tx, np.float64)
        self.u_rx = np.asarray(u_rx, np.float64)
        shape = (len(self.tx_points), len(self.rx_points), self.a.shape[-1])
        for name, arr, s in (('a', self.a, shape), ('tau', self.tau, shape),
                             ('u_tx', self.u_tx, shape + (3,)), ('u_rx', self.u_rx, shape + (3,))):
            if arr.shape != s:
                raise ValueError(f"{name}: {s} for {shape[0]} transmitter and {shape[1]} receiver anchors, not {arr.shape}")
        self.frequency = float(frequency)
        self.subcarriers = np.zeros(1) if subcarriers is None else np.asarray(subcarriers, np.float64).reshape(-1)
        self.reciprocal, self.max_distance, self.antenna_height = reciprocal, max_distance, antenna_height
        self.node_positions: dict = {}
        self.set_positions(node_positions or {})

    @classmethod
    def from_npz(
        cls,
        path: str, # .npz written by `save`
        **kwargs, # The other arguments of `RayTracedPathsFading`
    ) -> RayTracedPathsFading:
        "Load ray-traced paths saved with `save`."
        with np.load(path) as f:
            arrays = {k: f[k] for k in ('tx_points', 'rx_points', 'a', 'tau', 'u_tx', 'u_rx')}
            frequency = float(f['frequency'])
        return cls(**arrays, frequency=frequency, **kwargs)

    def save(
        self,
        path: str, # Where to write the .npz
    ) -> None:
        "Save the paths (e.g. traced on a GPU node, used anywhere)."
        np.savez_compressed(path, tx_points=self.tx_points, rx_points=self.rx_points, a=self.a, tau=self.tau,
                            u_tx=self.u_tx, u_rx=self.u_rx, frequency=self.frequency)

    def set_positions(
        self,
        positions: Mapping[NodeId, Sequence], # Position of (some of) the nodes: (x, y, z), or (x, y) with `antenna_height`
    ) -> None:
        "Where the nodes are now."
        for node, p in positions.items():
            p = np.asarray(p, np.float64).reshape(-1)
            if p.shape[0] == 2:
                if self.antenna_height is None:
                    raise ValueError(f"node {node!r} has a 2-D position: give the antenna_height")
                p = np.append(p, self.antenna_height)
            if p.shape[0] != 3:
                raise ValueError(f"node {node!r}: a position (x, y) or (x, y, z), not {tuple(p)}")
            self.node_positions[node] = p

    def _position(self, node: NodeId) -> np.ndarray:
        if node not in self.node_positions:
            raise KeyError(f"no position for node {node!r}: call set_positions first")
        return self.node_positions[node]

    @staticmethod
    def _nearest(points: np.ndarray, p: np.ndarray) -> tuple:
        d2 = ((points - p) ** 2).sum(-1)
        i = int(np.argmin(d2))
        return i, float(np.sqrt(d2[i]))

    def _paths(self, sender: NodeId, receiver: NodeId) -> tuple:
        "The anchors' paths of a link and how far each end is from its anchor; None if both ends are on the same anchor."
        ps, pr = self._position(sender), self._position(receiver)
        i, di = self._nearest(self.tx_points, ps)
        j, dj = self._nearest(self.rx_points, pr)
        forward = (max(di, dj), i, j, False)
        best = forward
        if self.reciprocal:                            # the receiver at a transmitter anchor, the path reversed
            i2, di2 = self._nearest(self.tx_points, pr)
            j2, dj2 = self._nearest(self.rx_points, ps)
            if max(di2, dj2) < forward[0]:
                best = (max(di2, dj2), i2, j2, True)
        far, i, j, reverse = best
        if self.max_distance is not None and far > self.max_distance:
            raise KeyError(f"link {sender!r} -> {receiver!r} is {far:.3f} m from the nearest anchors "
                           f"(max_distance {self.max_distance} m): no paths were ray-traced there")
        if np.array_equal(self.tx_points[i], self.rx_points[j]):
            return None                                # both on the same anchor: no paths were traced there
        a, tau = self.a[i, j], self.tau[i, j]
        if not reverse:                                # leaves the sender like the anchor path, arrives like it
            return a, tau, self.u_tx[i, j], ps - self.tx_points[i], self.u_rx[i, j], pr - self.rx_points[j]
        return a, tau, self.u_rx[i, j], ps - self.rx_points[j], self.u_tx[i, j], pr - self.tx_points[i]

    def cfr(
        self,
        sender: NodeId, # The transmitting node
        receiver: NodeId, # The receiving node
    ) -> np.ndarray:
        "Channel frequency response at the subcarriers (one value without subcarriers), at the nodes' exact positions."
        paths = self._paths(sender, receiver)
        if paths is None:
            return np.full(len(self.subcarriers), complex(math.inf))
        a, tau, u_dep, d_dep, u_arr, d_arr = paths
        live = a != 0
        a, tau, u_dep, u_arr = a[live], tau[live], u_dep[live], u_arr[live]
        length = tau * _C
        # the transmitter's move, mirrored like the path (u_dep onto -u_arr): its image's move
        n = u_dep + u_arr
        nn = (n * n).sum(-1, keepdims=True)
        image_move = d_dep - np.divide(2 * n * (n @ d_dep)[:, None], nn, out=np.zeros_like(n), where=nn > 1e-12)
        moved = np.linalg.norm(length[:, None] * u_arr + image_move - d_arr, axis=-1)
        dtau = (moved - length) / _C
        a = a * (length / moved) * np.exp(-2j * np.pi * self.frequency * dtau)
        return np.exp(-2j * np.pi * np.outer(self.subcarriers, tau + dtau)) @ a

    def path_gain(self, sender: NodeId, receiver: NodeId) -> float:
        "Linear path gain mean |H|^2 from `sender` to `receiver` at their current positions (inf if co-located)."
        if self._colocated(sender, receiver):
            return math.inf
        return float(np.mean(np.abs(self.cfr(sender, receiver)) ** 2))

    def _colocated(self, sender: NodeId, receiver: NodeId) -> bool:
        "At the same place, or nearer each other than the anchors (on the same anchor): a perfect link."
        return np.array_equal(self._position(sender), self._position(receiver)) or self._paths(sender, receiver) is None

    def response(self, transmission, link):
        "`cfr` of a transmission: H at each subcarrier (math.inf if co-located)."
        if self._colocated(transmission.sender, transmission.receiver):
            return np.full(len(self.subcarriers), complex(math.inf))
        return self.cfr(transmission.sender, transmission.receiver)

    def coefficient(self, transmission, link):
        "H at the carrier; with subcarriers, √(mean |H|²) (the same power). math.inf if co-located."
        if self._colocated(transmission.sender, transmission.receiver):
            return math.inf
        H = self.cfr(transmission.sender, transmission.receiver)
        return complex(H[0]) if H.shape[0] == 1 else complex(np.sqrt(np.mean(np.abs(H) ** 2)))

In [ ]:
show_doc(RayTracedPathsFading.cfr)

In [ ]:
show_doc(RayTracedPathsFading.set_positions)

In [ ]:
show_doc(RayTracedPathsFading.save)

In [ ]:
show_doc(RayTracedPathsFading.from_npz)

#### Tests

A geometry with an exact answer: a line-of-sight path and a reflection off a wall at `y = 3` (image method),
at 5.18 GHz. The paths are "ray-traced" at anchors every 5 cm, then the channel is computed at positions between
them and compared with the exact one.

In [ ]:
F = 5.18e9
WALL, GAMMA = 3.0, -0.7                                          # wall y = 3, reflection coefficient

def two_ray(pt, pr):
    "Exact paths from pt to pr: line of sight and the wall's reflection (image method)."
    pt, pr = np.asarray(pt, float), np.asarray(pr, float)
    mirror = lambda p: p * [1, -1, 1] + [0, 2 * WALL, 0]
    a, tau, ut, ur = [], [], [], []
    for gain, src, dst in ((1.0, pt, pr), (GAMMA, mirror(pt), pr)):
        length = np.linalg.norm(dst - src)
        a.append(gain * _C / F / (4 * np.pi * length) * np.exp(-2j * np.pi * F * length / _C))
        tau.append(length / _C)
        ur.append((src - dst) / length)                                  # towards the (image) source
    ut = [(pr - pt) / np.linalg.norm(pr - pt), (mirror(pr) - pt) / np.linalg.norm(mirror(pr) - pt)]
    return np.array(a), np.array(tau), np.array(ut), np.array(ur)

NO_PATHS = (np.zeros(2, complex), np.zeros(2), np.zeros((2, 3)), np.zeros((2, 3)))   # a point to itself

def trace(tx_points, rx_points):
    "Our 'ray tracer': the exact paths between every pair of anchors."
    out = {k: [] for k in ('a', 'tau', 'u_tx', 'u_rx')}
    for pt in tx_points:
        rows = [two_ray(pt, pr) if not np.array_equal(pt, pr) else NO_PATHS for pr in rx_points]
        for k, n in zip(out, range(4)):
            out[k].append([r[n] for r in rows])
    return {k: np.array(v) for k, v in out.items()}

exact_H = lambda pt, pr, f=np.zeros(1): np.exp(-2j * np.pi * np.outer(f, two_ray(pt, pr)[1])) @ two_ray(pt, pr)[0]

ap = np.array([[0.0, 0.0, 2.4]])                                # an access point
anchors = grid_points((2, 0), (3, 1), 0.05, height=0.4)         # a robot's area, every 5 cm
fading = RayTracedPathsFading(ap, anchors, **trace(ap, anchors), frequency=F, max_distance=0.1)
rng = np.random.default_rng(0)
robots = rng.uniform((2, 0), (3, 1), size=(200, 2))
err_plane, err_nearest = [], []
for xy in robots:
    pr = np.append(xy, 0.4)
    fading.set_positions({'ap': ap[0], 'robot': pr})
    truth = exact_H(ap[0], pr)[0]
    err_plane.append(abs(fading.cfr('ap', 'robot')[0] - truth) ** 2 / abs(truth) ** 2)
    nearest = anchors[np.argmin(((anchors - pr) ** 2).sum(-1))]
    err_nearest.append(abs(exact_H(ap[0], nearest)[0] - truth) ** 2 / abs(truth) ** 2)
print(f'error of H between 5 cm anchors: moved paths {10 * np.log10(np.mean(err_plane)):.0f} dB, '
      f'nearest anchor {10 * np.log10(np.mean(err_nearest)):.0f} dB')
assert np.mean(err_plane) < 1e-3 and np.mean(err_nearest) > 0.1
# at an anchor: exactly the traced channel; the reverse direction (robot -> access point) is the same channel
fading.set_positions({'robot': anchors[7]})
assert np.isclose(fading.cfr('ap', 'robot')[0], exact_H(ap[0], anchors[7])[0], rtol=1e-12)
assert np.isclose(fading.cfr('robot', 'ap')[0], fading.cfr('ap', 'robot')[0], rtol=1e-12)
t = Transmission(message=None, sender='ap', receiver='robot', start_time=0.0)
assert fading.response(t, None).shape == (1,) and NoFading(2j).response(t, None).tolist() == [2j]

In [ ]:
# both ends moving: agent-to-agent links, anchors every 10 cm on both sides
pts = grid_points((0, 0), (1, 1), 0.1, height=0.4)
agents = RayTracedPathsFading(pts, pts, **trace(pts, pts), frequency=F, antenna_height=0.4)
errs = []
for _ in range(200):
    p1, p2 = rng.uniform(0, 1, (2, 2))
    agents.set_positions({'a': p1, 'b': p2})                    # 2-D positions: at the antenna height
    if np.linalg.norm(p1 - p2) < 0.2:
        continue
    truth = exact_H(np.append(p1, 0.4), np.append(p2, 0.4))[0]
    errs.append(abs(agents.cfr('a', 'b')[0] - truth) ** 2 / abs(truth) ** 2)
print(f'agent-to-agent, 10 cm anchors on both ends: error {10 * np.log10(np.mean(errs)):.0f} dB')
assert np.mean(errs) < 1e-2
# wideband: 52 subcarriers of a 20 MHz channel
sc = (np.arange(52) - 25.5) * 312.5e3
wide = RayTracedPathsFading(ap, anchors, **trace(ap, anchors), frequency=F, subcarriers=sc)
pr = np.array([2.512, 0.333, 0.4])
wide.set_positions({'ap': ap[0], 'robot': pr})
H = wide.cfr('ap', 'robot')
assert H.shape == (52,) and np.allclose(H, exact_H(ap[0], pr, sc), rtol=0.05, atol=0)
assert abs(wide.path_gain('ap', 'robot') - np.mean(np.abs(exact_H(ap[0], pr, sc)) ** 2)) < 0.05 * wide.path_gain('ap', 'robot')

In [ ]:
import tempfile, os

In [ ]:
# errors: no position, too far from every anchor, 2-D without a height; co-located; save / load
t = Transmission(message=None, sender='ap', receiver='robot', start_time=0.0)
fading.set_positions({'robot': (9.0, 9.0, 0.4)})
try:
    fading.coefficient(t, None); raise AssertionError('no error')
except KeyError as e:
    assert 'max_distance' in str(e)
try:
    fading.set_positions({'x': (1.0, 2.0)}); raise AssertionError('no error')
except ValueError as e:
    assert 'antenna_height' in str(e)
fading.set_positions({'robot': ap[0]})
assert fading.coefficient(t, None) == math.inf
agents.set_positions({'a': (0.31, 0.52), 'b': (0.29, 0.49)})       # 3.6 cm apart, on the same 10 cm anchor
assert agents.path_gain('a', 'b') == math.inf and np.isinf(agents.cfr('a', 'b')).all()
with tempfile.TemporaryDirectory() as tmp:
    fading.save(os.path.join(tmp, 'paths.npz'))
    loaded = RayTracedPathsFading.from_npz(os.path.join(tmp, 'paths.npz'), node_positions={'ap': ap[0], 'robot': (2.51, 0.3, 0.4)})
    fading.set_positions({'robot': (2.51, 0.3, 0.4)})
    assert np.array_equal(loaded.cfr('ap', 'robot'), fading.cfr('ap', 'robot'))

With a channel, a robot moving continuously through the area:

In [ ]:
from comm_core.system import SystemLevelChannel
from comm_core.core import Message
from comm_core.topology import Link

In [ ]:
channel = SystemLevelChannel(fading=fading, noise_power=10 ** ((-94 - 30) / 10), seed=0)   # -94 dBm noise, 1 W
snr = []
for x in np.linspace(2.0, 3.0, 11):
    channel.set_positions({'ap': ap[0], 'robot': (x, 0.37, 0.4)})
    r = channel.transmit(Transmission(Message('ap', 'robot', None, size_bits=1000), 'ap', 'robot', 0.0), Link('ap', 'robot'))
    snr.append(round(r.snr_db, 1))
print('SNR along the way (dB):', snr)
assert all(s > 0 for s in snr)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()